<a href="https://colab.research.google.com/github/Megeeee/AgenticAI/blob/AntrophicAgent/nnUnetTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, torch, numpy as np, nibabel as nib, tempfile

os.environ['nnUNet_raw']          = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = '/content/nnUNet_results'

In [2]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [3]:
def init_nnUnet(tile_step_size: float = 0.5,
                 use_gaussian: bool = True,
                 use_mirroring: bool = True,
                model_training_output_dir  = "",
                use_folds = (0,),
                checkpoint_name = "checkpoint_final.pth"):
  model = nnUNetPredictor(
    tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
  model.initialize_from_trained_model_folder(model_training_output_dir,use_folds,checkpoint_name)
  return model

In [5]:
def segment(nnUnetPredictor,flair, t1, t1ce, t2) -> dict:
    """Run tumor segmentation on 4 modality paths, return mask path + volumes."""
    tmp_in, tmp_out = tempfile.mkdtemp(), tempfile.mkdtemp()
    for src, idx in zip([flair, t1, t1ce, t2], ['0000','0001','0002','0003']):
        nib.save(nib.load(src), os.path.join(tmp_in, f'case_{idx}.nii.gz'))

    nnUnetPredictor.predict_from_files(
        tmp_in, tmp_out, save_probabilities=False, overwrite=True,
        num_processes_preprocessing=1, num_processes_segmentation_export=1,
    )

    m = nib.load(os.path.join(tmp_out, 'case.nii.gz'))
    pred = m.get_fdata().astype(int)
    cm3 = float(np.prod(m.header.get_zooms()[:3])) / 1000

    return {
        'mask_path': os.path.join(tmp_out, 'case.nii.gz'),
        'whole_tumor_cm3':     round(float(np.isin(pred,[1,2,3]).sum())*cm3, 2),
        'tumor_core_cm3':      round(float(np.isin(pred,[1,3]).sum())*cm3, 2),
        'enhancing_tumor_cm3': round(float((pred==3).sum())*cm3, 2),
    }